# **응급상황 자동 인식 및 응급실 연계 서비스**
# **단계4 : 통합 - pipeline**

## **0.미션**

단계 4에서는, 단계1,2,3 에서 생성한 함수들을 모듈화하고, 단위 테스트 및 파이프라인 코드를 작성합니다.

* **미션6**
    * 단위 테스트
        * 각 기능(함수)에 대해 단계별로 테스트를 수행하며 오류를 해결합니다.
    * 파이프라인 구축
        * 단계1의 결과가 단계2 모델에 input이 되고, 모델의 예측 결과를 기반으로
        * 응급실 추천되도록
        * 조원들이 녹음한 음성 파일에 임의의 좌표(위도, 경도)값을 부여
            * 음성파일 이름과 좌표를 저장하는 별도 데이터셋 생성
        * 각 모듈을 연결하여 파이프라인 구성하는 ipynb 파일 생성



## **1.환경설정**

### (1) 경로 설정

구글 드라이브 연결

#### 1) 구글 드라이브 폴더 생성
* 새 폴더(project6_2)를 생성하고
* 제공 받은 파일을 업로드

#### 2) 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path = '/content/drive/MyDrive/# KT aivle school/# 수업 코드/#17 Mini Project 6th-2/취합/'

### (2) 라이브러리

#### 1) 필요한 라이브러리 설치

* requirements.txt 파일의 [경로 복사]를 한 후,
* 아래 경로에 붙여 넣기

In [38]:
# 경로 : /content/drive/MyDrive/project6_2/requirements.txt
# 경로가 다른 경우 아래 코드의 경로 부분을 수정하세요.

!pip install -r '/content/drive/MyDrive/# KT aivle school/# 수업 코드/#17 Mini Project 6th-2/취합/requirements.txt'

#### 2) 라이브러리 로딩

In [ ]:
#필요한 라이브러리 설치 및 불러우기
import os
import sys
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import openai
from openai import OpenAI
import json
import torch
import re
from haversine import haversine
from tqdm import tqdm
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
#path = '/content/drive/MyDrive/miniproj6/'
sys.path.append(path)


# 조에서 생성한 모듈 불러오기 -------------
from emergency import *

## **2. 단위 테스트**

* 세부사항 : 아래 단계별로 데이터가 순차적으로 처리되도록 단위 테스트를 진행합니다.

### (1) open ai key 등록

In [39]:
#각 모듈에서 불러와 사용하고 있어 생략하였습니다

### (2) audio to text

In [40]:
path

'/content/drive/MyDrive/# KT aivle school/# 수업 코드/#17 Mini Project 6th-2/취합/'

In [41]:
a2t = audio_to_text(path,'audio/audio1.mp3')
print("Audio2Text",a2t)

**************************************************
STT RESULT:  지금 아빠가 넘어졌어요. 머리에서 피가 나는데 숨은 쉬고 있어요. 지금 막 일어났어요. 근데 조금 어지럽다고 하네요. 네네, 계단에서 굴렀어요. 지금은 물 마시고 있는데, 이거 응급실로 가봐야 할까요? 피도 지금 머졌어요. 네네, 나이는 마흔아홉 살 이세요. 어떻게 해야 할지 모르겠어요.

**************************************************
Audio2Text 지금 아빠가 넘어졌어요. 머리에서 피가 나는데 숨은 쉬고 있어요. 지금 막 일어났어요. 근데 조금 어지럽다고 하네요. 네네, 계단에서 굴렀어요. 지금은 물 마시고 있는데, 이거 응급실로 가봐야 할까요? 피도 지금 머졌어요. 네네, 나이는 마흔아홉 살 이세요. 어떻게 해야 할지 모르겠어요.



### (3) text summary

In [43]:
summ,list = summ_list_extract(a2t)
print("SummedText",summ)

Text:  아빠가 넘어져 피가 나고 숨을 쉬는 상황에서 어지럽다는 증상이 나타남
##################################################
List:  ['지금', '아빠', '넘어졌어요', '머리', '피', '숨', '쉬고', '있어요', '지금', '막', '일어났어요', '조금', '어지럽다고', '하네요', '계단에서', '굴렀어요', '지금은', '물', '맨', '있는데', '응급실로', '가봐야', '할까요', '피도', '지금', '머졌어요', '나이는', '마흔아홉', '살', '어떻게', '해야', '할지', '모르겠어요']
##################################################
SummedText 아빠가 넘어져 피가 나고 숨을 쉬는 상황에서 어지럽다는 증상이 나타남


### (4) 응급실 등급분류

In [44]:
model = AutoModelForSequenceClassification.from_pretrained(path)
tokenizer = AutoTokenizer.from_pretrained(path)

In [45]:
predicted_class, probabilities = predict(summ, model, tokenizer)

print(f"예측된 클래스 이름: {predicted_class+1}등급")
print(f"클래스별 확률: {probabilities}")

예측된 클래스 이름: 2등급
클래스별 확률: tensor([[0.2665, 0.5230, 0.0821, 0.0467, 0.0818]])


### (5) 응급실추천

In [46]:
a = recommend_hospital(37.35,127.11)

100%|██████████| 10/10 [00:11<00:00,  1.12s/it]


## **3. 파이프라인**

* 세부사항
    * [2. 단계별 테스트] 의 내용을 순차적으로 정리합니다.
        * 데이터 처리 전 준비작업 : 한번 실행하면 되는 영역
            * 키, 데이터로딩
            * 모델/토크나이저 로딩
        * 입력값이 들어 왔을 때 출력값까지 처리되는 영역

In [48]:
def main(path, audio, location):
  model = AutoModelForSequenceClassification.from_pretrained(path)
  tokenizer = AutoTokenizer.from_pretrained(path)

  summ,list = summ_list_extract(audio_to_text(path,audio))
  predicted_class, probabilities = predict(summ, model, tokenizer)

  if predicted_class + 1 in [1, 2, 3]:
    print(f"예측된 클래스 이름: {predicted_class+1}등급")
    print(f"클래스별 확률: {probabilities}")
    idx = audio.split('/')[-1]
    call_lat,call_long = get_lat_long(idx, path + location)
    corr, name, m_list = recommend_hospital(call_lat,call_long)
    print(name[0])
    display(m_list[0])
    print(name[1])
    display(m_list[1])
    print(name[2])
    display(m_list[2])

### main 함수 사용!!
i = 2
audio = f'audio/audio{i}.mp3'
location = 'audio_location.xlsx'
main(path, audio, location)